# 01 - Exploratory Data Analysis (EDA)

Notebook phân tích dữ liệu gốc: kiểm tra kích thước, kiểu dữ liệu, missing values, thống kê mô tả, phân phối target, mất cân bằng lớp, tương quan và trực quan hóa các biến quan trọng.

## 1. Import thư viện

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load dữ liệu

In [ ]:
DATA_PATH = ROOT / "data" / "raw" / "diabetes_binary_health_indicators_BRFSS2015.csv"

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Kiểm tra thông tin tổng quan

In [ ]:
# Kich thuoc du lieu
print(f"So dong: {df.shape[0]:,}")
print(f"So cot: {df.shape[1]}")
print(f"\nSo dong: 253,680")
print(f"So cot: 22 cot gom 1 target va 21 feature")

In [ ]:
df.info()

In [ ]:
df.dtypes

## 4. Kiểm tra missing values

In [ ]:
missing_count = df.isnull().sum()
missing_percent = df.isnull().mean() * 100

missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

missing_report

In [ ]:
# Ket luan: Dataset sach, khong co missing values
print("Ket luan: Dataset sach, khong can xu ly missing values.")

## 5. Thống kê mô tả

In [ ]:
df.describe().T

In [ ]:
# Chu y cac bien lien tuc/ordinal quan trong
print("\nCac bien can chu y:")
print("- BMI: min={}, max={}, mean={:.1f}".format(df['BMI'].min(), df['BMI'].max(), df['BMI'].mean()))
print("- MentHlth: so ngay suc khoe tinh than khong tot trong 30 ngay")
print("- PhysHlth: so ngay suc khoe the chat khong tot trong 30 ngay")
print("- Age: nhom tuoi dang ordinal (1-13)")
print("- GenHlth: muc suc khoe tong quat dang ordinal (1-5)")

## 6. Đếm tần suất từng biến

### 6.1. Target variable

In [ ]:
print("Target Distribution:")
print(df["Diabetes_binary"].value_counts().sort_index())
print("\nPhan tram:")
print(df["Diabetes_binary"].value_counts(normalize=True) * 100)

In [ ]:
# Ve phan phoi target
target_counts = df["Diabetes_binary"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
sns.barplot(x=target_counts.index, y=target_counts.values)
plt.title("Target Distribution")
plt.xlabel("Diabetes_binary")
plt.ylabel("Count")
plt.xticks([0, 1], ["Non-diabetic (0)", "Diabetic (1)"])
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "target_distribution.png", dpi=150)
plt.show()

In [ ]:
# Nhan xet mat can bang lop
non_diabetic_pct = (df["Diabetes_binary"] == 0).mean() * 100
diabetic_pct = (df["Diabetes_binary"] == 1).mean() * 100
print(f"\nKet luan:")
print(f"- Non-diabetic: {non_diabetic_pct:.1f}%")
print(f"- Diabetic: {diabetic_pct:.1f}%")
print(f"\nDataset mat can bang nghiem trong.")
print(f"Accuracy co the gay hieu lanh neu model chi du doan day so 0.")
print(f"Can uu tien ROC-AUC, F1-score, Recall cho lop diabetic.")

### 6.2. Tần suất các biến categorical/binary/ordinal

In [ ]:
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].value_counts().sort_index())

## 7. Correlation Heatmap

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Xem feature lien quan manh voi Diabetes_binary
target_corr = corr["Diabetes_binary"].sort_values(ascending=False)
print("Feature lien quan voi Diabetes_binary:")
print(target_corr)

In [ ]:
# Cac bien thuong co lien quan dang chu y
print("\nCac bien co tuong quan manh nhat voi target:")
print("- GenHlth: suc khoe tong quat")
print("- HighBP: huyet ap cao")
print("- BMI: chi so khoi co the")
print("- DiffWalk: kho di lai")
print("- HighChol: cholesteron cao")
print("- Age: tuoi")
print("- HeartDiseaseorAttack: benh tim")
print("- PhysHlth: suc khoe the chat")

## 8. Histogram các biến liên tục/ordinal

In [ ]:
continuous_cols = ["BMI", "Age", "MentHlth", "PhysHlth"]

df[continuous_cols].hist(figsize=(12, 8), bins=30)
plt.suptitle("Histograms of Continuous / Ordinal Features")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "continuous_histograms.png", dpi=150)
plt.show()

In [ ]:
print("Nhan xet:")
print("- BMI phan phoi chuan, trung binh ~28")
print("- Age phan phoi deu cac nhom tuoi")
print("- MentHlth va PhysHlth lech phai, nhieu gia tri 0")

## 9. Boxplot theo Target

In [ ]:
continuous_cols = ["BMI", "Age", "MentHlth", "PhysHlth"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, col in zip(axes, continuous_cols):
    sns.boxplot(data=df, x="Diabetes_binary", y=col, ax=ax)
    ax.set_title(f"{col} by Diabetes_binary")
    ax.set_xlabel("Diabetes_binary")
    ax.set_ylabel(col)

plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "boxplots.png", dpi=150)
plt.show()

In [ ]:
print("Nhan xet:")
print("- Nhom diabetic thuong co BMI, Age, GenHlth, PhysHlth cao hon.")
print("- MentHlth co the lech phai vi nhieu nguoi co gia tri 0.")
print("- Mot so bien co outlier nhung khong can loai bo vi co the la ca that co y nghia du bao.")

## 10. Ket luan EDA

In [ ]:
print("""
KET LUAN EDA:
=============
1. Du lieu sach: 253,680 dong, 22 cot, khong co missing values.
2. Target mat can bang: ~86% non-diabetic, ~14% diabetic.
3. Cac feature quan trong nhat: GenHlth, HighBP, BMI, DiffWalk, HighChol, Age.
4. BMI phan phoi chuan, cac bien khac chu yeu la binary/ordinal.
5. Khong can xu ly missing values, outlier giu nguyen.

KHUYEN NGHI:
- Dung ROC-AUC, F1, Recall lam metric chinh thay vi Accuracy.
- Can xu ly mat can bang bang class_weight hoac SMOTE.
- Nhung buoc tiep theo: Feature Engineering, chia train/test, chuan hoa, train model.
""")